In [ ]:
# OpenCode / OpenAI-compatible notebook setup.
# Install once in the notebook's Python environment: pip install openai python-dotenv pyyaml
#
###################################################################################################
###################################################################################################
################################################################################################### Standard Modules
#
import os as _nb_os

from pathlib import Path as _NotebookPath

from dotenv import load_dotenv as _notebook_load_dotenv
#
###################################################################################################
###################################################################################################
################################################################################################### Notebook Paths
#
CHAPTER = "s03_permission"

PROJECT_ROOT = next(
    (
        directory
        for start in [_NotebookPath.cwd(), *_NotebookPath.cwd().parents]
        for directory in [start]
        if ((directory / CHAPTER / "code.py").is_file())
    ),
    None,
)

if (PROJECT_ROOT is None):

    raise (FileNotFoundError(
        f"Start Jupyter from the project root or {CHAPTER}/ "
        "so code.py can be located"
    ))
#

_notebook_load_dotenv(PROJECT_ROOT / ".env", override=True)
_nb_os.chdir(PROJECT_ROOT)

__file__ = str(PROJECT_ROOT / CHAPTER / "code.py")
#
###################################################################################################
###################################################################################################
################################################################################################### Interactive Entry Point
#
# Run All must not start a blocking CLI. Enable this explicitly if needed.
RUN_AGENT_INTERACTIVELY = False
#
###################################################################################################
###################################################################################################
################################################################################################### OpenCode Configuration
#
# The existing ANTHROPIC_* environment names remain supported.
_notebook_key   = (_nb_os.getenv("OPENCODE_API_KEY")  or  _nb_os.getenv("ANTHROPIC_API_KEY"))
_notebook_model = (_nb_os.getenv("OPENCODE_MODEL")  or  _nb_os.getenv("MODEL_ID"))
_notebook_url   = (
    _nb_os.getenv("OPENCODE_BASE_URL")
    or  _nb_os.getenv("ANTHROPIC_BASE_URL")
    or  "https://opencode.ai/inference/openai/v1"
).rstrip("/")

if (_notebook_url.endswith("/chat/completions")):

    _notebook_url = _notebook_url[:-len("/chat/completions")]
#

if (not _notebook_key):

    raise (RuntimeError(
        "Set OPENCODE_API_KEY (or ANTHROPIC_API_KEY) in the root .env"
    ))
#

if (not _notebook_model):

    raise (RuntimeError(
        "Set MODEL_ID (or OPENCODE_MODEL) in the root .env"
    ))
#

_nb_os.environ["OPENCODE_API_KEY"]   = _notebook_key
_nb_os.environ["OPENCODE_BASE_URL"]  = _notebook_url
_nb_os.environ["ANTHROPIC_BASE_URL"] = _notebook_url
_nb_os.environ["MODEL_ID"]          = _notebook_model

print(f"{CHAPTER}: OpenCode via OpenAI-compatible API, model={_notebook_model}")
#


In [ ]:
# OpenCode adapter: expose the Anthropic-shaped lesson API on top of OpenAI Chat Completions.
# The original .py remains untouched. This module mapping also supports s15/s16,
# which load earlier chapter .py files from disk via importlib.
import json as _compat_json
import sys as _compat_sys
import types as _compat_types
from openai import OpenAI as _OpenAI


def _compat_field(block, key, default=None):
    """Read a content field from either a provider object or a Python dict."""

    return block.get(key, default) if isinstance(block, dict) else getattr(block, key, default)


def _compat_text(value):
    """Convert Anthropic-style text blocks into text for Chat Completions."""

    if isinstance(value, str):
        return value
    if value is None:
        return ""
    if isinstance(value, list):
        return "\n".join(
            _compat_text(_compat_field(item, "text", _compat_field(item, "content", "")))
            for item in value
            if _compat_field(item, "type") != "tool_use"
        )
    if isinstance(value, dict):
        return _compat_text(value.get("text", value.get("content", _compat_json.dumps(value))))
    return str(value)


def _compat_history(messages, system=None):
    """Translate assistant tool calls and user tool results without losing call IDs."""

    result = []
    if system:
        result.append({"role": "system", "content": _compat_text(system)})

    for message in messages:
        role = _compat_field(message, "role")
        content = _compat_field(message, "content", "")
        if role == "assistant":
            if not isinstance(content, list):
                result.append({"role": "assistant", "content": _compat_text(content)})
                continue
            text_blocks = []
            tool_calls = []
            for block in content:
                kind = _compat_field(block, "type")
                if kind == "tool_use":
                    arguments = _compat_field(block, "input", {})
                    tool_calls.append({
                        "id": _compat_field(block, "id"),
                        "type": "function",
                        "function": {
                            "name": _compat_field(block, "name"),
                            "arguments": _compat_json.dumps(arguments, ensure_ascii=False),
                        },
                    })
                elif kind == "text":
                    text_blocks.append(_compat_field(block, "text", ""))
            converted = {"role": "assistant", "content": "\n".join(text_blocks) or None}
            if tool_calls:
                converted["tool_calls"] = tool_calls
            if converted["content"] is None and not tool_calls:
                converted["content"] = ""
            result.append(converted)
        elif role == "user" and isinstance(content, list):
            user_text = []
            tool_results = []
            for block in content:
                if _compat_field(block, "type") == "tool_result":
                    tool_results.append({
                        "role": "tool",
                        "tool_call_id": _compat_field(block, "tool_use_id"),
                        "content": _compat_text(_compat_field(block, "content", "")),
                    })
                else:
                    user_text.append(_compat_text(block))
            result.extend(tool_results)
            if user_text:
                result.append({"role": "user", "content": "\n".join(user_text)})
        else:
            result.append({"role": role, "content": _compat_text(content)})
    return result


class _NotebookMessages:
    """Translate lessons' messages.create calls to OpenAI-compatible requests."""

    def __init__(self, client):
        self._client = client

    def create(self, *, model, messages, system=None, tools=None, max_tokens=None, **kwargs):
        """Return Anthropic-shaped text/tool blocks with original call identifiers."""

        request = {"model": model, "messages": _compat_history(messages, system)}
        if tools:
            request["tools"] = [
                {
                    "type": "function",
                    "function": {
                        "name": tool["name"],
                        "description": tool.get("description", ""),
                        "parameters": tool.get(
                            "input_schema", {"type": "object", "properties": {}}
                        ),
                    },
                }
                for tool in tools
            ]
        if max_tokens is not None:
            request["max_tokens"] = max_tokens

        response = self._client.chat.completions.create(**request)
        choice = response.choices[0]
        reply = choice.message
        blocks = []
        if reply.content:
            blocks.append(_compat_types.SimpleNamespace(type="text", text=reply.content))
        for call in reply.tool_calls or []:
            try:
                arguments = _compat_json.loads(call.function.arguments or "{}")
                if not isinstance(arguments, dict):
                    arguments = {}
            except (ValueError, TypeError):
                arguments = {}
            blocks.append(_compat_types.SimpleNamespace(
                type="tool_use", id=call.id, name=call.function.name, input=arguments
            ))
        usage = getattr(response, "usage", None)
        return _compat_types.SimpleNamespace(
            content=blocks,
            stop_reason=(
                "max_tokens" if choice.finish_reason == "length"
                else "tool_use" if reply.tool_calls
                else "end_turn"
            ),
            usage=_compat_types.SimpleNamespace(
                input_tokens=getattr(usage, "prompt_tokens", 0) or 0,
                output_tokens=getattr(usage, "completion_tokens", 0) or 0,
            ),
        )


class _NotebookAnthropic:
    """Provide the old lesson client interface over the configured OpenCode endpoint."""

    def __init__(self, base_url=None, **kwargs):
        url = _nb_os.getenv("OPENCODE_BASE_URL") or base_url or _notebook_url
        self._client = _OpenAI(api_key=_notebook_key, base_url=url)
        self.messages = _NotebookMessages(self._client)


_anthropic_compat = _compat_types.ModuleType("anthropic")
_anthropic_compat.Anthropic = _NotebookAnthropic
_compat_sys.modules["anthropic"] = _anthropic_compat


In [ ]:
#!/usr/bin/env python3
"""
s03_permission.py - Permission System

Three gates inserted before tool execution:

    Gate 1: Hard deny list (rm -rf /, sudo, ...)
    Gate 2: Rule matching (write outside workspace? destructive cmd?)
    Gate 3: User approval (pause and wait for confirmation)

    +----------+      +-------+      +--------------+      +---------------+
    |   User   | ---> |  LLM  | ---> | Permission   | ---> | Tool Dispatch |
    |  prompt  |      |       |      | 1. deny list |      | execute       |
    +----------+      +---+---+      | 2. rules     |      +-------+-------+
                          ^          | 3. approval  |              |
                          |          +------+-------+              |
                          |                 | deny                 |
                          |                 v                      v
                          |          +-------------------------------+
                          +----------+ tool_result: denied or output |
                                     +-------------------------------+

Only one line added to the agent loop:

    if not check_permission(block):
        continue

Builds on s02 (multi-tool). Usage:

    python s03_permission/code.py
    Needs: pip install anthropic python-dotenv + ANTHROPIC_API_KEY in .env
"""


In [ ]:

#
###################################################################################################
###################################################################################################
################################################################################################### Standard Modules
#
import os
import re
import subprocess
from pathlib import Path

try:

    import readline
    readline.parse_and_bind('set bind-tty-special-chars off')
    readline.parse_and_bind('set input-meta on')
    readline.parse_and_bind('set output-meta on')
    readline.parse_and_bind('set convert-meta off')
#
except ImportError:

    pass
#

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv(override=True)
if (os.getenv("ANTHROPIC_BASE_URL")):

    os.environ.pop("ANTHROPIC_AUTH_TOKEN", None)
#

WORKDIR = Path.cwd()
client  = Anthropic(base_url=os.getenv("ANTHROPIC_BASE_URL"))
MODEL   = os.environ["MODEL_ID"]

ENVIRONMENT_PROMPT = (
    ("Windows: the bash tool runs through cmd.exe; use cmd.exe syntax, not Unix "
    "Bash or PowerShell syntax, and prefer dedicated file tools for file operations")
    if (os.name == "nt")
    else ("Unix-like: the bash tool runs the system shell")
)

SYSTEM = (
    f"You are a coding agent at {WORKDIR}. Environment: {ENVIRONMENT_PROMPT}. "
    "All destructive operations require user approval."
)


In [ ]:


#
###################################################################################################
###################################################################################################
################################################################################################### From s02: tool implementations
#

def run_bash(command:str) -> str:

    try:

        r   = subprocess.run(
            command,
            shell          = True,
            cwd            = WORKDIR,
            capture_output = True,
            text           = True,
            errors         = "replace",
            timeout        = 120,
        )
        out = (r.stdout + r.stderr).strip()
        return ((out[:50000]) if (out) else ("(no output)"))
    #
    except subprocess.TimeoutExpired:

        return ("Error: Timeout (120s)")
    #
#


def run_read(path:str, limit:int|None=None) -> str:

    try:

        lines = (WORKDIR / path).resolve().read_text(encoding="utf-8").splitlines()
        if (limit  and  limit < len(lines)):

            lines = lines[:limit] + [f"... ({len(lines) - limit} more lines)"]
        #
        return ("\n".join(lines))
    #
    except Exception as e:

        return (f"Error: {e}")
    #
#


def run_write(path:str, content:str) -> str:

    try:

        file_path = (WORKDIR / path).resolve()
        file_path.parent.mkdir(parents=True, exist_ok=True)
        file_path.write_text(content, encoding="utf-8")
        return (f"Wrote {len(content)} bytes to {path}")
    #
    except Exception as e:

        return (f"Error: {e}")
    #
#


def run_edit(path:str, old_text:str, new_text:str) -> str:

    try:

        file_path = (WORKDIR / path).resolve()
        text      = file_path.read_text(encoding="utf-8")
        if (old_text not in text):

            return (f"Error: text not found in {path}")
        #
        file_path.write_text(text.replace(old_text, new_text, 1), encoding="utf-8")
        return (f"Edited {path}")
    #
    except Exception as e:

        return (f"Error: {e}")
    #
#


def run_glob(pattern:str) -> str:

    import glob as g
    try:

        matches = sorted({
            match for match in g.glob(
            pattern,
            root_dir  = WORKDIR,
            recursive = True,
        )
            if (WORKDIR / match).resolve().is_relative_to(WORKDIR)
        })
        shown   = matches[:200]
        if (len(matches) > 200):

            shown.append("... (more matches omitted; narrow the pattern)")
        #
        return (("\n".join(shown)) if (shown) else ("(no matches)"))
    #
    except Exception as e:

        return (f"Error: {e}")


In [ ]:
    #
#


#
###################################################################################################
###################################################################################################
################################################################################################### From s02 (unchanged): tool definitions and dispatch
#

TOOLS = [
    {
        "name"         : "bash",
        "description"  : "Run a shell command.",
        "input_schema" : {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]},
    },
    {
        "name"         : "read_file",
        "description"  : "Read file contents.",
        "input_schema" : {"type": "object", "properties": {"path": {"type": "string"}, "limit": {"type": "integer"}}, "required": ["path"]},
    },
    {
        "name"         : "write_file",
        "description"  : "Write content to a file.",
        "input_schema" : {"type": "object", "properties": {"path": {"type": "string"}, "content": {"type": "string"}}, "required": ["path", "content"]},
    },
    {
        "name"         : "edit_file",
        "description"  : "Replace exact text in a file once.",
        "input_schema" : {"type": "object", "properties": {"path": {"type": "string"}, "old_text": {"type": "string"}, "new_text": {"type": "string"}}, "required": ["path", "old_text", "new_text"]},
    },
    {
        "name"         : "glob",
        "description"  : "Find files matching a glob pattern; ** matches recursively.",
        "input_schema" : {"type": "object", "properties": {"pattern": {"type": "string"}}, "required": ["pattern"]},
    },
]

TOOL_HANDLERS = {
    "bash"       : run_bash,
    "read_file"  : run_read,
    "write_file" : run_write,
    "edit_file"  : run_edit,
    "glob"       : run_glob,
}


In [ ]:


#
###################################################################################################
###################################################################################################
################################################################################################### New in s03: three-gate permission pipeline
#

# Gate 1: Hard deny list - always forbidden
DENY_LIST = ["rm -rf /", "sudo", "shutdown", "reboot", "mkfs", "dd if=", "> /dev/sda"]

def check_deny_list(command:str) -> str|None:

    for pattern in DENY_LIST:

        if (pattern in command):

            return (f"Blocked: '{pattern}' is on the deny list")
        #
    #
    return (None)
#


# Gate 2: Rule matching - context-dependent checks
DESTRUCTIVE_COMMAND_WORD = re.compile(
    r"(?i)(?:^|[;&|()\n])\s*(?:rm|del)(?=\s|$|[;&|()])"
)


def contains_destructive_command(command:str) -> bool:

    return (bool(DESTRUCTIVE_COMMAND_WORD.search(command)))
#


PERMISSION_RULES = [
    {
        "tools"   : ["read_file", "write_file", "edit_file"],
        "check"   : lambda args: not (WORKDIR / args.get("path", "")).resolve().is_relative_to(WORKDIR),
        "message" : "Writing outside workspace",
    },
    {
        "tools"   : ["bash"],
        "check"   : lambda args: contains_destructive_command(args.get("command", ""))  or
     any(kw in args.get("command", "") for kw in ["rm ", "> /etc/", "chmod 777"]),
        "message" : "Potentially destructive command",
    },
]

def check_rules(tool_name:str, args:dict) -> str|None:

    for rule in PERMISSION_RULES:

        if (tool_name in rule["tools"]  and  rule["check"](args)):

            return (rule["message"])
        #
    #
    return (None)
#


# Gate 3: User approval - wait for confirmation after rule match
def ask_user(tool_name:str, args:dict, reason:str) -> str:

    print(f"\n\033[33m[permission] {reason}\033[0m")
    print(f"   Tool: {tool_name}({args})")
    choice = input("   Allow? [y/N] ").strip().lower()
    return (("allow") if (choice in ("y", "yes")) else ("deny"))
#


# Pipeline: all three gates chained
def check_permission(block) -> bool:

    if (block.name == "bash"):

        reason = check_deny_list(block.input.get("command", ""))
        if (reason):

            print(f"\n\033[31m[blocked] {reason}\033[0m")
            return (False)
        #
    #
    reason = check_rules(block.name, block.input)
    if (reason):

        decision = ask_user(block.name, block.input, reason)
        if (decision == "deny"):

            return (False)
        #
    #
    return (True)


In [ ]:
#


#
###################################################################################################
###################################################################################################
################################################################################################### Agent loop: same as s02, with check_permission() inserted
#

def agent_loop(messages:list):

    while (True):

        response = client.messages.create(
            model      = MODEL,
            system     = SYSTEM,
            messages   = messages,
            tools      = TOOLS,
            max_tokens = 8000,
        )
        messages.append({"role": "assistant", "content": response.content})

        tool_calls = [
            block for block in response.content if block.type == "tool_use"
        ]
        if (not tool_calls):

            return
        #

        results = []
        for block in tool_calls:

            print(f"\033[36m> {block.name}\033[0m")

            # s03 change: run through permission pipeline before executing
            if (not check_permission(block)):

                results.append({
                    "type"        : "tool_result",
                    "tool_use_id" : block.id,
                    "content"     : "Permission denied.",
                })
                continue
            #

            handler = TOOL_HANDLERS.get(block.name)
            output  = (handler(**block.input)) if (handler) else (f"Unknown: {block.name}")
            print(str(output)[:200])
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": output})
        #

        messages.append({"role": "user", "content": results})
    #
#


if (RUN_AGENT_INTERACTIVELY):

    print("s03: Permission")
    print("Enter a question, press Enter to send. Type q to quit.\n")

    history = []
    while (True):

        try:

            # \001/\002 tell Readline the ANSI escapes have zero display width.
            query = input("\001\033[36m\002s03 >> \001\033[0m\002")
        #
        except (EOFError, KeyboardInterrupt):

            break
        #
        if (query.strip().lower() in ("q", "exit", "")):

            break
        #
        history.append({"role": "user", "content": query})
        agent_loop(history)
        for block in history[-1]["content"]:

            if (getattr(block, "type", None) == "text"):

                print(block.text)
            #
        #
        print()
    #
#
